In [ ]:
from platform import python_version
print(python_version())

In [ ]:
!echo $CONDA_DEFAULT_ENV

### Install Cellpose-SAM

https://pypi.org/project/cellpose/

https://github.com/MouseLand/cellpose/blob/main/notebooks/run_Cellpose-SAM.ipynb

In [ ]:
!nvidia-smi

In [ ]:
!nvcc --version

In [ ]:
# !pip3 install scikit-learn
# !pip3 install seaborn
# !pip3 install plotly

In [ ]:
import os, sys, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

sys.path.insert(1, '../src/')

from Basic import *
from image_lib import *

pd.set_option("display.precision", 3)
from IPython.display import display, HTML
display(HTML("<style>:root { --jp-notebook-max-width: 100% !important; }</style>"))

#### Check if colab notebook instance has GPU access

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models

from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
import numpy as np

In [ ]:
reload_np = False

if reload_np:
    del(np)
    import numpy as np
np.__version__

In [ ]:
# !pip3 install scispacy==0.6.2 --force-reinstall

In [ ]:
# !pip install "numpy<2.0" --force-reinstall
# !pip3 install opencv-python==4.9.0.80
# !pip3 install grad-cam

In [ ]:
print(">>> PyTorch version", torch.__version__)

if torch.cuda.is_available():
    print(">> current_device:", torch.cuda.current_device())
    print(">> Device:", torch.cuda.get_device_name(0))
    print(">> CUDA:", torch.version.cuda)
else:
    print(">>> only CPU")
    

In [ ]:
root0 = "../../colaboracoes/deOcesano"
os.listdir(root0)

In [ ]:
cp = Cellpose(root0=root0, verbose=True)

In [ ]:
cp.root_samples

In [ ]:
cp.root_crop

In [ ]:
plates = os.listdir(cp.root_samples)
plates

In [ ]:
i=0
plate=plates[i]
cp.create_roots_plate(plate, verbose=True)

plate

In [ ]:
exps = [x for x in os.listdir(cp.root_plate) if os.path.isdir(os.path.join(cp.root_plate, x))]
exps

In [ ]:
j=2
experiment = exps[j]

cp.create_roots_experiment(experiment, verbose=True)

experiment

### Data origin

In [ ]:
i=1
plate = plates[i]
cp.create_roots_plate(plate, verbose=False)

exps = [x for x in os.listdir(cp.root_plate) if os.path.isdir(os.path.join(cp.root_plate, x))]

print(plate, '\n', exps)

In [ ]:
cp.root_image

In [ ]:
fnames = [x for x in os.listdir(cp.root_image) if x.endswith('tif')]
fnames[0]

In [ ]:
i=0
filefig = os.path.join(cp.root_image, fnames[i])
img = io.imread(filefig)

fig = plt.figure(figsize=(5,5))
(width, height, channels) = img.shape
print(width, height)
plt.imshow(img);

In [ ]:
img.shape

In [ ]:
image = cp.read_PIL_image(fnames[i], cp.root_image).convert('RGB')
image.size

In [ ]:
type_image = 'tif'; num_test_imgs = 15
ncrop=5

dir_origins = ['10% SFB', '1% SFB', '1% SFB and IL-1B']
class_names = ['10pct', '1pct', '1pct_il1b']
dir_names = ['10perc', '1perc', 'IL1B']

model_name='resnet18'

root_data, dic_root_train, dic_root_test = \
cp.set_data_origin_and_create_roots_to_train_and_test(model_name=model_name,  crop_segment='crop', ncrop=ncrop, image_example=image, 
                                                      class_names=class_names, dir_origins=dir_origins, dir_names=dir_names, verbose=True)

In [ ]:
cp.model_fname, cp.model_table

In [ ]:
cp.clean_data(verbose=True)

In [ ]:
cp.root_data

### Sampling

In [ ]:
root_train_case = os.path.join(cp.root_train)
root_test_case = os.path.join(cp.root_test)

In [ ]:
ncrop

In [ ]:
max_images=1200
perc_train = .60
perc_test = 1-perc_train

s_end = f'_ncrop_{5}.png'
print(">>> s_end", s_end, '\n\n')

dic={}; icount=-1
for i in range(len(class_names)):
    root_data_case = os.path.join(cp.root_data, dir_origins[i])
    class_name = class_names[i]

    fnames = [x for x in os.listdir(root_data_case) if x.endswith(s_end)]
    random.shuffle(fnames)
    
    n = len(fnames)

    if n < max_images:
        maxi = n
        print(">>> max_images", maxi)
    else:
        maxi = max_images
        print(">>> max_images", maxi)

    samples = fnames[:maxi]
    n_samples = len(samples)

    n_train_samples = int(n_samples*perc_train)
    train_samples = samples[:n_train_samples]
    test_samples  = samples[n_train_samples:]

    n_test_samples  = len(test_samples)

    icount += 1
    
    dic[icount] = {}
    dic2 = dic[icount]
    dic2['class_name'] = class_name
    dic2['n'] = n
    dic2['perc_train'] = perc_train
    dic2['n_samples'] = n_samples
    dic2['n_train_samples'] = n_train_samples
    dic2['n_test_samples'] = n_test_samples
    dic2['root_data'] = root_data_case
    dic2['train_samples'] = train_samples
    dic2['test_samples'] = test_samples
    
    print(f"{i}) {class_name:12} n={n:5} -> {root_data_case}")

df = pd.DataFrame(dic).T
df

In [ ]:
mat = df.iloc[0].train_samples
if isinstance(mat, str):
    mat = eval(mat)

mat[:10]

In [ ]:
def copy_data_train_test(class_names:List, dir_origins:List, dir_names:List, max_images:int=1200, perc_train:float=.60, 
                         type_image:str='png', verbose:bool=False) -> Tuple[bool, pd.DataFrame]:

    if not cp.clean_data(verbose=verbose):
        return False, pd.DataFrame()
    
    # perc_test = 1-perc_train
    s_end = f'_ncrop_{5}.{type_image}'
    
    dic={}; icount=-1
    for i, class_name in enumerate(class_names):
        dir_origin = dir_origins[i]
        dir_target = dir_names[i]

        root_data_case = os.path.join(cp.root_data, dir_origin)
        root_target_train = create_dir(cp.root_train, dir_target)
        root_target_test  = create_dir(cp.root_test,  dir_target)
    
        fnames = [x for x in os.listdir(root_data_case) if x.endswith(s_end)]
        
        n = len(fnames)
    
        if n < max_images:
            maxi = n
            print(">>> max_images", maxi)
        else:
            maxi = max_images
            print(">>> max_images", maxi)
    
        random.shuffle(fnames)
        samples = fnames[:maxi]
        n_samples = len(samples)
    
        n_train_samples = int(n_samples*perc_train)
        train_samples = samples[:n_train_samples]
        test_samples  = samples[n_train_samples:]
    
        n_test_samples  = len(test_samples)
    
        icount += 1
        
        dic[icount] = {}
        dic2 = dic[icount]
        dic2['class_name'] = class_name
        dic2['n'] = n
        dic2['perc_train'] = perc_train
        dic2['n_samples'] = n_samples
        dic2['n_train_samples'] = n_train_samples
        dic2['n_test_samples'] = n_test_samples
        dic2['root_data'] = root_data_case
        dic2['root_target_train'] = root_target_train
        dic2['root_target_test'] = root_target_test
        dic2['train_samples'] = train_samples
        dic2['test_samples'] = test_samples
        
        print(f"{i}) {class_name:12} n={n:5} -> {root_data_case}")

    df = pd.DataFrame(dic).T

    for i, row in df.iterrows():
        row = df.iloc[i]
        class_name = row.class_name
        root_data = row.root_data

        #------------ train data -------------------------------
        root_target_train = row.root_target_train
        train_samples = row.train_samples
        train_samples = train_samples if isinstance(train_samples, list) else eval(train_samples)
        
        print(f">>> moving class '{class_name}': {len(train_samples)} train samples from \n{root_data} to {root_target_train}")
        for fname in train_samples:
            filename = os.path.join(root_data, fname)
            try:
                shutil.copy(filename, root_target_train)
            except:
                print(f"Error: moving {filename} to {root_target_train}")
                return False, df
                      

        #------------ test data --------------------------------
        root_target_test  = row.root_target_test
        test_samples = row.test_samples
        test_samples = test_samples if isinstance(test_samples, list) else eval(test_samples)
        
        print(f">>> moving class '{class_name}': {len(test_samples)} test samples from \n{root_data} to {root_target_test}")
        for fname in test_samples:
            filename = os.path.join(root_data, fname)
            try:
                shutil.copy(filename, root_target_test)
            except:
                print(f"Error: moving {filename} to {root_target_test}")
                return False, df

    return True, df


In [ ]:
class_names, dir_origins, dir_names

In [ ]:
ret, df = copy_data_train_test(class_names=class_names, dir_origins=dir_origins, dir_names=dir_names, max_images=1200, perc_train=.60, type_image='png')
print(ret)

In [ ]:
df

In [ ]:
print(df.iloc[0].root_data)
print(df.iloc[0].root_target_train)
print(df.iloc[0].root_target_test)

### ImageDataset

In [ ]:
img.shape, img.size

In [ ]:
ncrop = 5
size_x = int(width/5)
size_y = int(height/5)
size_x, size_y, 3

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize(size=(size_x, size_y)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize(size=(size_x, size_y)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

In [ ]:
df.columns

In [ ]:
df

In [ ]:
class ImageDataset(torch.utils.data.Dataset):

    def __init__(self, df:pd.DataFrame, transform, train_or_test:str='train'):
        
        self.transform = transform

        self.class_names = np.unique(df.class_name)
        self.class_to_idx = {name: i for i, name in enumerate(self.class_names)}

        self.samples = []
        for _, row in df.iterrows():

            class_name = row.class_name
            label = self.class_to_idx[class_name]

            if train_or_test == 'train':
                samples = row.train_samples
                root = row.root_target_train
            elif train_or_test == 'test':
                samples = row.test_samples
                root = row.root_target_test
            else:
                print("Error: train or test?")
                raise ValueError('\n\n--------------- stop --------------\n')

            samples = samples if isinstance(samples, list) else eval(samples)

            # a FLAT LIST OF (image_path, label)
            # print(">>>", label, len(samples))
            
            self.samples += [ (img_name, root, label) for img_name in samples]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        img_name, root, label = self.samples[index]

        image = cp.read_PIL_image(img_name, root).convert("RGB")
        x = self.transform(image)

        return x, label


# Prepare DataLoader

In [ ]:
train_dataset = ImageDataset(df=df, transform=train_transform, train_or_test='train')

In [ ]:
test_dataset = ImageDataset(df=df, transform=test_transform, train_or_test='test')

In [ ]:
train_dataset.samples[0], len(train_dataset.samples)

In [ ]:
test_dataset.samples[0], len(test_dataset.samples)

In [ ]:
batch_size = 10

dl_train = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
dl_test  = torch.utils.data.DataLoader(test_dataset,  batch_size=batch_size, shuffle=True)

print('Number of training batches', len(dl_train))
print('Number of test batches', len(dl_test))

In [ ]:
for x in dl_train:
    print(x[0].shape, x[1])
    break

In [ ]:
images, labels = next(iter(dl_train))
print(images.shape)
print(labels.shape)

# Data Visualization

In [ ]:
class_names = train_dataset.class_names

def show_images(images, labels, preds, fontsize=14, figsize=(12, 20)):
    plt.figure(figsize=figsize )
    for i, image in enumerate(images):
        plt.subplot(5, 2, i + 1, xticks=[], yticks=[])
        image = image.numpy().transpose((1, 2, 0))
        mean = np.array([0.485, 0.456, 0.406])
        std  = np.array([0.229, 0.224, 0.225])
        image = image * std + mean
        image = np.clip(image, 0., 1.)
        plt.imshow(image)

        # print(i, preds[i], labels[i])
        col = 'red' if (preds[i] != labels[i]) else 'green'
            
        plt.xlabel(f'{class_names[int(labels[i].numpy())]}', fontsize=fontsize)
        plt.ylabel(f'{class_names[int(preds[i].numpy())]}', color=col, fontsize=fontsize)
    plt.tight_layout()
    plt.show()

# Creating the Model

In [ ]:
train_dataset.class_names

In [ ]:
train_dataset.class_to_idx

### Parameters

In [ ]:
lr=1e-5
n_classes = len(class_names)

### Model: ResNet18

In [ ]:
x[0].shape, size_x, size_y

In [ ]:
lr=1e-4
weight_decay=lr/2
n_classes = len(class_names)

model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
model.fc = nn.Linear(in_features=model.fc.in_features, out_features= n_classes)

model.fc.in_features, model.fc.out_features

In [ ]:
try:
    device = "cuda"
    model = model.to(device)
    print("Ok CUDA")
except:
    print("Error: device must be CUDA - however, it is not available")

In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

from torch.optim.lr_scheduler import StepLR
# scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.1)
scheduler = StepLR(optimizer, step_size=5, gamma=0.5)

In [ ]:
def show_preds(model, fontsize=14, figsize=(12, 20)):
    model.eval()
    images, labels = next(iter(dl_test))
    images = images.to(device)
    
    print(images.shape, images[0].shape, labels.shape, labels[0])
    outputs = model(images)
    _, preds = torch.max(outputs, 1)
    # print(preds)
    show_images(images.cpu(), labels, preds.cpu(), fontsize=fontsize, figsize=figsize)

In [ ]:
show_preds(model, fontsize=12, figsize=(5, 10))

# Training the Model

In [ ]:
cp.root_data

In [ ]:
root_best = os.path.join(cp.root_data, 'best_model')
create_dir(root_best)

fname0=f'best_model_weights_for_{plate}_crop_{ncrop}.pt'
fname_best_model= os.path.join(root_best, fname0)

fname_best_model

### Eval accuracy

In [ ]:
def evaluate_loss_accuracy(model, loader):
    model.eval()
    correct = 0
    total = 0
    loss_sum = 0

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            with torch.amp.autocast('cuda'):
                outputs = model(images)
                loss = nn.CrossEntropyLoss()(outputs, labels)
                preds = outputs.argmax(1)

            loss_sum += loss.item()
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    return loss_sum / len(loader), correct / total    

### Treinamento com Mixed Precision (AMP) — 2× mais rápido

#### Early Stopping + Best Weights / training
#### Avoid to train to much and worsing the accuracy

In [ ]:
fname_best_model0 = cp.model_fname
fname_best_table0 = cp.model_table

fname_best_model = os.path.join(root_best, fname_best_model0)
fname_best_table = os.path.join(root_best, fname_best_table0)

fname_best_model, fname_best_table

In [ ]:
scaler = torch.amp.GradScaler('cuda')

best_acc = 0

In [ ]:
train_losses = []
val_losses = []
val_accuracies = []

In [ ]:
epochs=100
early_stop_patience=20

wait = 0

for epoch in range(epochs):
    model.train()
    epoch_loss = 0.0
    start = time.time()

    for images, labels in dl_train:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        with torch.amp.autocast('cuda'):
            outputs = model(images)
            loss = loss_fn(outputs, labels) # cross-entropy

        scaler.scale(loss).backward()

        # Gradient clipping
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        
        scaler.step(optimizer)
        scaler.update()

        epoch_loss += loss.item()

    # acc = eval_accuracy(model, dl_test, device)
    train_loss = epoch_loss / len(dl_train)
    val_loss, val_acc = evaluate_loss_accuracy(model, dl_test)

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    val_accuracies.append(val_acc)

    elapsed = time.time() - start
    print(f"{epoch+1}/{len(train_losses)}) loss: {val_loss:.4f}, Accuracy: {val_acc:.4f}, LR: {lr:.2e} Time: {elapsed:.1f}s")

    if val_acc > best_acc:
        best_acc = val_acc
        wait = 0
        try:
            torch.save(model.state_dict(), fname_best_model)
            print(f"Model saved at: '{fname_best_model}'")
        except:
            print(f"Error: model not saved at: '{fname_best_model}'")
    else:
        wait += 1
        if wait >= early_stop_patience:
            print("Early stopping! Next epoch ...")
            break
        print(">>> wait", wait)

print(f"\nTraining finished. Best accuracy = {best_acc:.4f}")
        

In [ ]:
len(val_losses), len(train_losses)

In [ ]:
fname_best_model, fname_best_table

In [ ]:
dfacc = pd.DataFrame({'train_loss': train_losses, 'val_loss': val_losses, 'accuracy': val_accuracies})

pdwritecsv(dfacc, fname_best_table0, root_best, verbose=True)

dfacc.tail(10)

In [ ]:
epochs_range = range(1, len(train_losses) + 1)
plt.figure(figsize=(12,5))

#------ loss plot ------------------
plt.subplot(1, 2, 1)
plt.plot(epochs_range, train_losses, marker='o', label='Train Loss')
plt.plot(epochs_range, val_losses, marker='o', label='Validation Loss')
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training & Validation Loss")
plt.legend()
plt.grid(True)

#----- accuracy plot ---------------
plt.subplot(1, 2, 2)
plt.plot(epochs_range, val_accuracies, marker='o', label='Validation Accuracy')
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Validation Accuracy")
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

### Recover best model

In [ ]:
fname_best_model

In [ ]:
new_model = model
new_model.load_state_dict(torch.load(fname_best_model, map_location=device))
# new_model.eval()

In [ ]:
show_preds(new_model, fontsize=12, figsize=(5, 10))